# Tarea 2 · Parte 1 – Corpus y extracción

- **Fecha de extracción:** 25/10/2026
- **Fuentes y licencias:** Wikimedia Commons (dominio público, CC0, CC BY 2.0/3.0, CC BY-SA 3.0/4.0) para los 25 documentos y 12 entrevistas; OpenSLR 73 (CC BY-SA 4.0) para 6 clips leídos. Ningún archivo es de los usados en clase. Detalle por archivo en `datos/fuentes.csv`.
- **OCR:** PaddleOCR 3.7.0 · modelo PP-OCRv6 (detector + reconocedor, engine=transformers, GPU MPS)
- **ASR:** faster-whisper 1.2.1 · modelo `small` (Systran/faster-whisper-small, int8, CPU, VAD activado, idioma `es`)
- **Diarización:** WeSpeaker ResNet34 (pyannote) + clustering jerárquico (`voz.diarizar`)
- **Entorno:** proyecto uv (`pyproject.toml` + `uv.lock`), Python 3.12

In [1]:
import os, sys, time, json, hashlib, warnings, re, shutil
from pathlib import Path

os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"   # antes de importar PaddleOCR
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"                # si una operación no existe en MPS, usa CPU
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import requests
import soundfile as sf
import torch
import paddleocr, faster_whisper
from paddleocr import PaddleOCR
from PIL import Image

import voz
from voz import SR, Transcriptor, cargar_audio, diarizar, a_srt, a_texto, a_tabla, seudonimizar, tramos_de_voz

pio.renderers.default = "vscode+png"   # interactivo en VS Code y PNG para que se vea en GitHub

DATOS = Path("datos")
RAW_DOC, RAW_AUD = DATOS / "raw" / "documentos", DATOS / "raw" / "audio"
RAW_DOC.mkdir(parents=True, exist_ok=True); RAW_AUD.mkdir(parents=True, exist_ok=True)

print(f"PaddleOCR {paddleocr.__version__} · faster-whisper {faster_whisper.__version__} · PyTorch {torch.__version__}")
print("MPS disponible:", torch.backends.mps.is_available())


PaddleOCR 3.7.0 · faster-whisper 1.2.1 · PyTorch 2.14.1
MPS disponible: True


In [12]:
from urllib.parse import urlparse

API = "https://commons.wikimedia.org/w/api.php"
AGENTE = {"User-Agent": "tarea2-ds-python/2026 (https://github.com/jyauric/python-actividades)"}

def commons_api(**params):
    "Consulta la API de Wikimedia Commons con User-Agent y una pausa de cortesía."
    r = requests.get(API, params={"action": "query", "format": "json", **params}, headers=AGENTE, timeout=60)
    r.raise_for_status()
    time.sleep(1.5)
    return r.json()["query"]

def archivos_de_categoria(categoria, n=100):
    "Números de página (pageid) de los archivos de una categoría de Commons."
    q = commons_api(list="categorymembers", cmtitle=f"Category:{categoria}", cmtype="file", cmlimit=n)
    return [m["pageid"] for m in q["categorymembers"]]

CATEGORIAS = ["Banknotes of Peru", "Documents of Peru", "Posters of Peru",
              "Signs in Lima", "Street signs in Lima", "Baptism certificates of Peru"]
candidatos = sorted({pid for c in CATEGORIAS for pid in archivos_de_categoria(c)})
print(len(candidatos), "archivos candidatos")

244 archivos candidatos


In [13]:
def metadatos(pageids, ancho=None):
    "Licencia, autor, tamaño y enlace de descarga de cada archivo, según su página de Commons."
    filas = []
    for i in range(0, len(pageids), 20):
        extra = {"iiurlwidth": ancho} if ancho else {}       # ancho: pide una versión reducida de la imagen
        q = commons_api(pageids="|".join(map(str, pageids[i:i + 20])), prop="imageinfo",
                        iiprop="size|url|extmetadata", iiextmetadatafilter="LicenseShortName|Artist", **extra)
        for pg in q["pages"].values():
            ii = pg["imageinfo"][0]
            em = ii["extmetadata"]
            limpio = lambda k: re.sub(r"<[^>]+>", "", em.get(k, {}).get("value", "")).strip()
            filas.append({"pageid": pg["pageid"], "ancho": ii["width"], "alto": ii["height"],
                          "licencia": limpio("LicenseShortName"), "autor": limpio("Artist")[:60],
                          "url_descarga": ii.get("thumburl", ii["url"])})
    return pd.DataFrame(filas)

cand = metadatos(candidatos, ancho=1600)
LIBRES = r"Public domain|CC0|CC BY"                  # dominio público, CC0, CC BY y CC BY-SA
cand = cand[cand.licencia.str.contains(LIBRES)].reset_index(drop=True)
print(len(cand), "candidatos con licencia libre")
cand.licencia.value_counts()

244 candidatos con licencia libre


licencia
CC BY-SA 4.0     125
CC0               35
Public domain     31
CC BY-SA 2.0      20
CC BY 2.0         15
CC BY-SA 3.0      11
CC BY 3.0          3
CC BY 4.0          3
CC BY 3.0 de       1
Name: count, dtype: int64

### Selección de los 25 documentos

La búsqueda en 6 categorías de Wikimedia Commons devolvió 244 candidatos con licencia libre.
La preselección de 25 se hizo con apoyo del asistente de IA (Claude Code), con este criterio:

- 8 **impresos** nítidos, 8 **fotos** con ángulo, sombra o reflejos y 9 **manuscritos**;
- texto en español legible para una persona;
- ninguno de los documentos usados en clase.

Cada archivo se identifica por su número de página de Commons (`pageid`) y se cita con la URL `https://commons.wikimedia.org/?curid=…`, porque varios títulos contienen nombres de personas.
La licencia y el autor se toman de la API de Commons (no se escriben a mano), y la selección se verificó revisando las 25 imágenes.


In [17]:
SELECCION = {   # tipo → números de página de Commons (pageid)
    "impreso": [2121996,     # billete de 1 sol (s. XIX)
                2229021,     # billete de 5 soles (s. XIX)
                2263189,     # billete de 100 pesos (1867)
                2044186,     # billete de 40 centavos, Banco de Arequipa (1873)
                2284104,     # billete de 1 inca
                7279749,     # himno de Chanchamayo
                111181262,   # titular sobre Indecopi
                175594467],  # carta mecanografiada de la Presidencia (1956)
    "foto": [154062977, 154063152, 154063179,   # letreros con el antiguo nombre de la calle (Lima)
             124562195,      # panel informativo de una huaca
             150442442,      # cartel de rutas en la Panamericana Sur
             95621317,       # letrero del bosque El Olivar
             84154317,       # banner contra el dengue (Madre de Dios)
             153256484],     # hoja de servicios naval enmarcada, bajo vidrio
    "manuscrito": [160615606, 160615612, 160615607,   # Libro de Bautismos de Lampa (1655–1722, 1832, 1871)
                   15315035,     # recibo telegráfico de 1915 llenado a mano
                   67887441,     # partida de bautismo (s. XVIII)
                   190398555,    # partida de bautismo (1897)
                   131142697,    # afiches escritos a mano por estudiantes
                   131142702,    # carteles escritos a mano por estudiantes
                   5100641],     # tarjeta de circulación de 1921 llenada a mano
}
inventario = pd.DataFrame([(t, pid) for t, lista in SELECCION.items() for pid in lista], columns=["tipo", "pageid"])
inventario = inventario.merge(cand[["pageid", "licencia", "autor", "url_descarga"]], on="pageid", how="left")
assert inventario.licencia.notna().all(), "Hay un archivo elegido que no está entre los candidatos con licencia libre"
# 🔒 En dos archivos de dominio público, el campo «autor» de Commons repite datos del propio documento
# (la persona inscrita o su familia): se reemplaza por la institución que emitió el documento.
AUTOR_CORREGIDO = {190398555: "Parroquia de San Clemente de Pisco (registro parroquial)",
                   5100641: "Prefectura de Lima (tarjeta oficial de 1921)"}
inventario["autor"] = inventario.pageid.map(AUTOR_CORREGIDO).fillna(inventario.autor)
inventario["fuente_url"] = "https://commons.wikimedia.org/?curid=" + inventario.pageid.astype(str)
inventario.insert(0, "id", [f"doc_{i:03d}" for i in range(1, len(inventario) + 1)])   # ids neutrales
COLS_INV = ["id", "tipo", "pageid", "licencia", "autor", "fuente_url"]               # sin url_descarga: lleva el título
inventario[COLS_INV].to_csv(DATOS / "inventario_documentos.csv", index=False)
inventario.tipo.value_counts()          # 8 impreso, 8 foto, 9 manuscrito

tipo
manuscrito    9
impreso       8
foto          8
Name: count, dtype: int64

In [ ]:
from IPython.display import HTML

HTML(inventario.assign(imagen=inventario.url_descarga.map(lambda u: f'<img src="{u}" width="160">'))
     [["id", "tipo", "imagen", "licencia"]].to_html(escape=False, index=False))

In [19]:
def descargar(url, destino):
    "Baja un archivo de Wikimedia Commons. Si ya está en disco, no hace nada."
    if not destino.exists():
        r = requests.get(url, headers=AGENTE, timeout=120)
        r.raise_for_status()
        destino.write_bytes(r.content)
        time.sleep(1.5)                      # cortesía con el servidor (1 a 2 s)
    return destino

inventario["archivo"] = [str(descargar(f.url_descarga, RAW_DOC / f"{f.id}{Path(urlparse(f.url_descarga).path).suffix.lower()}"))
                         for f in inventario.itertuples()]

### La GPU de tu Mac

PaddleOCR nació sobre **PaddlePaddle**, el framework de aprendizaje profundo de Baidu, y
PaddlePaddle en una Mac **solo usa la CPU**. Desde la versión 3.5, PaddleOCR puede ejecutar
los mismos modelos con **PyTorch** (`engine="transformers"`), y PyTorch sí sabe hablar con la
GPU de los chips de Apple a través de **MPS** (*Metal Performance Shaders*).

Primero averiguamos qué tiene esta máquina:

In [20]:
if torch.backends.mps.is_available():
    DISPOSITIVO = "mps"       # GPU de un chip Apple (M1 en adelante)
elif torch.cuda.is_available():
    DISPOSITIVO = "cuda"      # GPU NVIDIA
else:
    DISPOSITIVO = "cpu"

print(f"PaddleOCR {paddleocr.__version__} · PyTorch {torch.__version__}")
print(f"Dispositivo elegido: {DISPOSITIVO}")

PaddleOCR 3.7.0 · PyTorch 2.14.1
Dispositivo elegido: mps


In [21]:
def redes(obj, _prof=0, _vistos=None):
    "Busca, dentro de un modelo de PaddleOCR, las redes neuronales de PyTorch."
    _vistos = set() if _vistos is None else _vistos
    if id(obj) in _vistos or _prof > 7:
        return
    _vistos.add(id(obj))
    if isinstance(obj, torch.nn.Module):
        yield obj
        return
    for hijo in (vars(obj).values() if hasattr(obj, "__dict__") else []):
        for x in (hijo.values() if isinstance(hijo, dict) else [hijo]):
            if not isinstance(x, (str, int, float, bool, type(None), list, tuple, set)):
                yield from redes(x, _prof + 1, _vistos)


PAPELES = {"Layout": "detector de bloques", "Det": "detector de texto",
           "Rec": "reconocedor de texto", "VL": "modelo de visión-lenguaje"}


def a_gpu(modelo, dispositivo=DISPOSITIVO):
    "Mueve todas las redes del modelo al dispositivo elegido e informa qué movió."
    for red in redes(modelo):
        red.to(dispositivo)
        clase = type(red).__name__
        papel = next((p for pista, p in PAPELES.items() if pista in clase), clase)
        millones = sum(p.numel() for p in red.parameters()) / 1e6
        print(f"  {papel:<26} {millones:6.1f} millones de parámetros → {dispositivo}")
    return modelo

In [22]:
def leer(ruta, modelo=None):
    "OCR de una imagen. Devuelve una tabla con una fila por cada línea de texto."
    r = (ocr if modelo is None else modelo).predict(str(ruta))[0]
    cajas = np.asarray(r["rec_boxes"]).reshape(-1, 4)          # x0, y0, x1, y1 de cada línea
    return pd.DataFrame({
        "texto": r["rec_texts"],
        "confianza": np.round(r["rec_scores"], 3),
        "x0": cajas[:, 0], "y0": cajas[:, 1], "x1": cajas[:, 2], "y1": cajas[:, 3],
    })

In [23]:
OPCIONES = dict(
    use_doc_orientation_classify=False,   # no girar la página completa
    use_doc_unwarping=False,              # no "planchar" hojas dobladas
    use_textline_orientation=False,       # no girar cada línea
)
ocr = PaddleOCR(engine="transformers", **OPCIONES)   # PyTorch, para poder usar la GPU de la Mac
a_gpu(ocr, "mps")
MODELO_OCR = f"PaddleOCR {paddleocr.__version__} PP-OCRv6 (engine=transformers, MPS, sin orientación ni unwarping)"

CRUDO = RAW_DOC / "crudo"; CRUDO.mkdir(exist_ok=True)
filas = []
for f in inventario.itertuples():
    r = ocr.predict(str(f.archivo))[0]
    r.save_to_json(str(CRUDO))            # respaldo crudo: textos, cajas y confianzas
    r.save_to_img(str(CRUDO))             # imagen con las cajas detectadas
    filas.append({
        "id": f.id, "archivo": f.archivo, "tipo": f.tipo,
        "fuente": f"{f.fuente_url} ({f.licencia})",
        "texto_ocr": " ".join(r["rec_texts"]),
        "modelo_ocr": MODELO_OCR,
        "n_lineas": len(r["rec_texts"]),
        "confianza_mediana": float(np.median(r["rec_scores"])) if len(r["rec_scores"]) else 0.0,
    })
documentos = pd.DataFrame(filas)
documentos.groupby("tipo")[["n_lineas", "confianza_mediana"]].median()

Creating model: ('PP-OCRv6_medium_det', None, 'transformers')
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/josselinyauri/.paddlex/official_models/PP-OCRv6_medium_det_safetensors`.
W1025 22:29:47.434000 9855 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 350/350 [00:00<00:00, 11508.63it/s]
Creating model: ('PP-OCRv6_medium_rec', None, 'transformers')
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/josselinyauri/.paddlex/official_models/PP-OCRv6_medium_rec_safetensors`.
Loading weights: 100%|██████████| 269/269 [00:00<00:00, 12117.71it/s]


  detector de texto            22.0 millones de parámetros → mps
  reconocedor de texto         19.1 millones de parámetros → mps


Connecting to https://paddle-model-ecology.bj.bcebos.com/paddlex/PaddleX3.0/fonts/simfang.ttf ...
[==================================================] 100.00%


,n_lineas,confianza_mediana
tipo,,
foto,11.5,0.966088
impreso,16.0,0.939521
manuscrito,16.0,0.851281
